In [17]:
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, TimestampType 
from pyspark.sql.functions import create_map, lit

In [ ]:
import json
from pathlib import Path 

config_path = (
    Path.cwd()
    / ".." / ".." / ".."
    / "runtime_config" / "config.json"
).resolve()

with config_path.open("r", encoding="utf-8") as file:
    config = json.load(file)

print(config)

{'dev': {'citibike_catalog': 'citibike_dev'}, 'test': {'citibike_catalog': 'citibike_test'}}


In [3]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.serverless().getOrCreate()

In [ ]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")
env = dbutils.widgets.get("env")

In [ ]:
schema = StructType([
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("ride_id", StringType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("start_lat", DecimalType(10, 6), True),
    StructField("start_lng", DecimalType(10, 6), True),
    StructField("end_lat", DecimalType(10, 6), True),
    StructField("end_lng", DecimalType(10, 6), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("member_casual", StringType(), True),
])

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"/Volumes/{config[env]['citibike_catalog']}/00_landing/source_citibike_data/JC-202503-citibike-tripdata.csv")
)

In [ ]:
df = df.withColumn(
    "metadata",
    create_map(
        lit("pipeline_id"), lit(pipeline_id),
        lit("run_id"), lit(run_id),
        lit("task_id"), lit(task_id),
        lit("processed_timestamp"), lit(processed_timestamp)
    )
)

In [ ]:
df.write.\
mode("overwrite").\
option("overwriteSchema", "true").\
saveAsTable(f"{config[env]['citibike_catalog']}.01_bronze.jc_citibike")